# Dataset completo: auditoría reproducible
Fuentes originales en `datasets/tax-document-dataset-v0.1`; tareas ejecutables en `datasets/tax-document-eval-v1`.
Este notebook no llama modelos. Los controles del evaluador no sustituyen la revisión documental. Instalar las dependencias de `tests/requirements.txt` en el kernel antes de ejecutarlo.

In [ ]:
from pathlib import Path
import sys, json, subprocess
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets/tax-document-eval-v1/tasks.json").is_file())
sys.path.insert(0, str(root))
dataset = root / "datasets/tax-document-eval-v1"
audit = json.loads((dataset / "audit.json").read_text())
{key: audit[key] for key in ("documents", "cases", "tasks", "fields", "document_formats", "unobservable_aggregate_fields")}

In [ ]:
# Hashes de todas las fuentes y reconstrucción exacta de tareas, valores y evidencias.
result = subprocess.run([sys.executable, "scripts/prepare_full_dataset.py", "--check"], cwd=root, check=True, capture_output=True, text=True)
json.loads(result.stdout)

In [ ]:
# Pruebas negativas del evaluador y sumas independientes por expediente.
# Son controles de contrato y aritmética; la evidencia visual está en los artefactos.
result = subprocess.run([sys.executable, "-m", "unittest", "tests.datasets.test_full_dataset", "-v"], cwd=root, capture_output=True, text=True)
print(result.stdout + result.stderr)
assert result.returncode == 0

In [ ]:
review = json.loads((root / "artifacts/2026-09-13/full-dataset/independent-preparation-review.json").read_text())
review["flag_amount_tensions"]

Las respuestas XML calculadas, las 39 plantillas/referencias oficiales y los 77 documentos ExtractBench no adquiridos no se puntúan. Todo el conjunto es de desarrollo; los documentos duplicados comparten source_group_id. Ver `docs/FULL_DATASET_READINESS.md` para ejecutar los harnesses.